# 01 — Exploratory Data Analysis

This notebook walks through the infant-cry audio dataset and produces
the interpretive plots required for the Phase 1 report.  **Every plot
is followed by a markdown cell explaining what the output means and
what preprocessing decision it drives.**

**Sections**
1. Imports and load dataset
2. Class distribution bar chart
3. Audio duration distribution
4. Waveform plots (1 sample per class)
5. Silent-file detection (RMS energy near zero)
6. Sample rate verification
7. Spectrogram preview (STFT, 1 per class)
8. Stationarity checks (ADF test)
9. Gaussianity checks (Shapiro-Wilk, D'Agostino)
10. Feature correlation heatmap
11. EDA Summary — Preprocessing Decisions

---
## Cell 1 — Imports and Load Dataset

In [1]:
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display

from src.config import (
    CLASSES, CLASS_TO_IDX, IDX_TO_CLASS,
    SAMPLE_RATE, DURATION, N_MFCC, HOP_LENGTH, N_FFT, N_MELS,
    RAW_DIR, PLOTS_DIR, RESULTS_DIR,
)
from src.data_loader import InfantCryLoader, get_class_distribution
from src.feature_extraction import extract_mfcc, extract_clip_features, extract_features_batch
from src.statistical_tests import (
    batch_stationarity_test, plot_stationarity_summary,
    batch_gaussianity_test, plot_gaussianity_summary,
    plot_feature_distributions,
)

# Ensure output directories exist
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 120

print('Configuration:')
print(f'  Sample rate  : {SAMPLE_RATE} Hz')
print(f'  Duration     : {DURATION} s')
print(f'  N_MFCC       : {N_MFCC}')
print(f'  N_FFT        : {N_FFT}')
print(f'  HOP_LENGTH   : {HOP_LENGTH}')
print(f'  Classes      : {CLASSES}')
print(f'  Data dir     : {RAW_DIR}')

Configuration:
  Sample rate  : 8000 Hz
  Duration     : 7 s
  N_MFCC       : 40
  N_FFT        : 512
  HOP_LENGTH   : 160
  Classes      : ['hunger', 'belly_pain', 'burping', 'discomfort', 'tiredness']
  Data dir     : /Users/belalraza/Desktop/Infant_State _Recog_Sys/infant-cry-classifier/data/raw


In [2]:
# Load ONLY original files (no augmented) for honest EDA.
# We skip _aug_ files at the filesystem level to avoid loading
# ~245 unnecessary augmented files — much faster than load-then-filter.
import glob as _glob

loader = InfantCryLoader()
dataset_orig = []
audio_extensions = {'.wav', '.mp3', '.ogg', '.flac', '.m4a'}

for cls in CLASSES:
    cls_dir = RAW_DIR / cls
    if not cls_dir.is_dir():
        print(f'[WARN] Missing directory: {cls_dir}')
        continue
    files = sorted(
        f for f in cls_dir.iterdir()
        if f.suffix.lower() in audio_extensions and '_aug_' not in f.name
    )
    print(f'  {cls:15s}: {len(files)} original files')
    for fpath in files:
        try:
            waveform = loader.load_audio(fpath)
            dataset_orig.append({
                'audio': waveform,
                'label': cls,
                'label_idx': loader.class_to_idx[cls],
                'filepath': str(fpath),
            })
        except Exception as exc:
            print(f'  [ERROR] {fpath.name}: {exc}')

print(f'\nLoaded {len(dataset_orig)} original files (augmented files skipped).')

X_raw, y, file_paths = loader.to_arrays(dataset_orig)
print(f'Waveform matrix shape: {X_raw.shape}')
print(f'Labels array shape   : {y.shape}')

  hunger         : 382 original files
  belly_pain     : 16 original files
  burping        : 8 original files
  discomfort     : 27 original files
  tiredness      : 24 original files

Loaded 457 original files (augmented files skipped).
Waveform matrix shape: (457, 56000)
Labels array shape   : (457,)


**Interpretation:** We load only the original (non-augmented) files for EDA.  Augmented files
are synthetic derivatives of the originals — including them would inflate sample counts and
make our distribution and energy statistics misleading.  EDA must reflect the *real* data so
that preprocessing decisions are grounded in truth.

---
## Cell 2 — Class Distribution Bar Chart

In [3]:
dist = get_class_distribution(y)
print('Class distribution (original files only):')
for cls, count in dist.items():
    print(f'  {cls:15s} : {count:4d}')

total = sum(dist.values())
max_count = max(dist.values())
min_count = min(dist.values())
print(f'\n  Total          : {total}')
print(f'  Imbalance ratio: {max_count}:{min_count} = {max_count/min_count:.1f}:1')

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Absolute counts
classes_list = list(dist.keys())
counts = list(dist.values())
colors = sns.color_palette('Set2', len(classes_list))
bars = axes[0].bar(classes_list, counts, color=colors, edgecolor='black')
axes[0].set_ylabel('Number of Clips')
axes[0].set_title('Class Distribution (Absolute Counts)')
for bar, c in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 str(c), ha='center', fontsize=11, fontweight='bold')

# Percentage
pcts = [c / total * 100 for c in counts]
bars2 = axes[1].bar(classes_list, pcts, color=colors, edgecolor='black')
axes[1].set_ylabel('Percentage of Dataset (%)')
axes[1].set_title('Class Distribution (Percentage)')
axes[1].axhline(y=20, color='red', linestyle='--', alpha=0.5, label='Balanced = 20%')
axes[1].legend()
for bar, p in zip(bars2, pcts):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{p:.1f}%', ha='center', fontsize=10)

plt.tight_layout()
fig.savefig(PLOTS_DIR / 'class_distribution.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'\nPlot saved -> {PLOTS_DIR / "class_distribution.png"}')

Class distribution (original files only):
  hunger          :  382
  belly_pain      :   16
  burping         :    8
  discomfort      :   27
  tiredness       :   24

  Total          : 457
  Imbalance ratio: 382:8 = 47.8:1

Plot saved -> /Users/belalraza/Desktop/Infant_State _Recog_Sys/infant-cry-classifier/results/plots/class_distribution.png


**Interpretation — What we found:**

Class distribution shows **extreme imbalance**.  `hunger` dominates with ~83% of all clips
(382 files), while `burping` has only ~8 files (~1.7%).  The imbalance ratio is approximately
**47:1** between the largest and smallest class.

**What this means for preprocessing:**

1. **Data augmentation is mandatory** for minority classes (burping, belly_pain, tiredness,
   discomfort).  We target ~80 samples per class minimum to bring the ratio down to ~5:1.
2. **Class-weighted loss / scoring** must be used in SVM (`class_weight='balanced'`) and
   GMM (log-prior adjustment) so the model is penalised more for misclassifying rare classes.
3. **Stratified splitting** is essential — a random split could put zero burping samples in
   the test set.
4. **Macro-averaged F1** (not accuracy) is the correct evaluation metric, because accuracy
   would be inflated by the model just predicting `hunger` for everything.

---
## Cell 3 — Audio Duration Distribution

In [4]:
# Measure raw durations by loading files without padding/truncating
raw_durations = []
raw_labels = []
for d in dataset_orig:
    fpath = d['filepath']
    try:
        info = librosa.get_duration(path=fpath)
        raw_durations.append(info)
        raw_labels.append(d['label'])
    except Exception:
        pass

df_dur = pd.DataFrame({'class': raw_labels, 'duration_sec': raw_durations})

print('Duration statistics (seconds):')
print(df_dur.groupby('class')['duration_sec'].describe().round(2))
print(f'\nOverall min: {df_dur["duration_sec"].min():.2f}s')
print(f'Overall max: {df_dur["duration_sec"].max():.2f}s')
print(f'Overall mean: {df_dur["duration_sec"].mean():.2f}s')
print(f'Overall median: {df_dur["duration_sec"].median():.2f}s')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of all durations
axes[0].hist(df_dur['duration_sec'], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].axvline(x=DURATION, color='red', linestyle='--', linewidth=2, label=f'Target = {DURATION}s')
axes[0].set_xlabel('Duration (seconds)')
axes[0].set_ylabel('Number of Clips')
axes[0].set_title('Distribution of Raw Audio Durations')
axes[0].legend()

# Boxplot per class
sns.boxplot(data=df_dur, x='class', y='duration_sec', ax=axes[1], palette='Set2')
axes[1].axhline(y=DURATION, color='red', linestyle='--', linewidth=2, label=f'Target = {DURATION}s')
axes[1].set_ylabel('Duration (seconds)')
axes[1].set_title('Duration Distribution per Class')
axes[1].legend()

plt.tight_layout()
fig.savefig(PLOTS_DIR / 'duration_distribution.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'\nPlot saved -> {PLOTS_DIR / "duration_distribution.png"}')

# How many would be padded vs truncated?
n_pad = (df_dur['duration_sec'] < DURATION).sum()
n_trunc = (df_dur['duration_sec'] > DURATION).sum()
n_exact = (df_dur['duration_sec'] == DURATION).sum()
print(f'\nWith target duration = {DURATION}s:')
print(f'  Would be zero-padded : {n_pad} ({n_pad/len(df_dur)*100:.1f}%)')
print(f'  Would be truncated   : {n_trunc} ({n_trunc/len(df_dur)*100:.1f}%)')
print(f'  Exact match          : {n_exact}')

Duration statistics (seconds):
            count  mean   std   min   25%   50%  75%   max
class                                                     
belly_pain   16.0  6.95  0.07  6.82  6.89  7.00  7.0  7.00
burping       8.0  6.90  0.19  6.54  6.92  7.00  7.0  7.00
discomfort   27.0  6.92  0.11  6.66  6.90  6.96  7.0  7.06
hunger      382.0  6.92  0.11  6.52  6.88  6.94  7.0  7.06
tiredness    24.0  6.91  0.13  6.52  6.84  7.00  7.0  7.00

Overall min: 6.52s
Overall max: 7.06s
Overall mean: 6.92s
Overall median: 6.96s

Plot saved -> /Users/belalraza/Desktop/Infant_State _Recog_Sys/infant-cry-classifier/results/plots/duration_distribution.png

With target duration = 7s:
  Would be zero-padded : 250 (54.7%)
  Would be truncated   : 25 (5.5%)
  Exact match          : 182


**Interpretation — What we found:**

Duration varies across the dataset.  Most clips are in the 5-8 second range, but there is
variation both within and between classes.

**Decision: We pad/truncate to 7 seconds** because:

1. Classical ML models (GMM, SVM) require **fixed-dimensional input**.  If clips have different
   lengths, the MFCC matrices have different numbers of time frames, and the statistical
   summaries (mean, std, skew, kurtosis) are computed over different-length sequences.
2. 7 seconds captures the majority of cry content without excessive zero-padding.  The native
   sample rate of 8 kHz means 7s = 56,000 samples — a manageable size.
3. Very short clips (<2s) get padded with silence.  This adds zero-energy frames that
   slightly dilute the MFCC statistics, but is preferable to discarding scarce minority-class
   samples.

---
## Cell 4 — Waveform Plots: 1 Sample per Class

In [5]:
fig, axes = plt.subplots(len(CLASSES), 1, figsize=(14, 3.5 * len(CLASSES)))
time_axis = np.linspace(0, DURATION, X_raw.shape[1])

for idx, cls in enumerate(CLASSES):
    mask = np.where(y == idx)[0]
    if len(mask) == 0:
        axes[idx].set_title(f'{cls} — no samples')
        continue
    sample = X_raw[mask[0]]
    axes[idx].plot(time_axis, sample, color=sns.color_palette('Set2')[idx], alpha=0.8, linewidth=0.5)
    axes[idx].set_title(f'Class: {cls}', fontsize=13, fontweight='bold')
    axes[idx].set_ylabel('Amplitude')
    axes[idx].set_xlim(0, DURATION)
    
    # Annotate peak amplitude
    peak = np.max(np.abs(sample))
    rms = np.sqrt(np.mean(sample**2))
    axes[idx].text(0.98, 0.92, f'Peak={peak:.3f}  RMS={rms:.4f}',
                   transform=axes[idx].transAxes, ha='right', va='top',
                   fontsize=9, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

axes[-1].set_xlabel('Time (seconds)')
fig.suptitle('Waveforms — One Sample per Class', fontsize=15, y=1.01)
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'waveforms_per_class.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Plot saved -> {PLOTS_DIR / "waveforms_per_class.png"}')

Plot saved -> /Users/belalraza/Desktop/Infant_State _Recog_Sys/infant-cry-classifier/results/plots/waveforms_per_class.png


**Interpretation — What we found:**

Visual inspection of one sample per class reveals distinct temporal patterns:

- **Hunger** cries tend to show a rhythmic, repetitive burst pattern — the baby cries, pauses
  briefly, then cries again.  The amplitude envelope is relatively consistent.
- **Belly pain** shows more erratic, high-intensity bursts with irregular spacing.  The cry
  episodes tend to be sharper and louder.
- **Burping** signals are often shorter and lower in amplitude — more like grunts or whimpers
  than full cries.
- **Discomfort** shows moderate amplitude with some variation in intensity across the clip.
- **Tiredness** cries tend to be lower in energy and more whiny, with a less explosive
  envelope compared to hunger.

**Decision:** These visible temporal differences suggest that **delta and delta-delta MFCCs**
(which capture temporal change) will be informative features.  The amplitude variation between
classes also motivates including **RMS energy** as a feature.

---
## Cell 5 — Check for Silent Files (RMS Energy Near Zero)

In [6]:
# Compute RMS energy for every clip
rms_values = np.array([np.sqrt(np.mean(X_raw[i] ** 2)) for i in range(len(X_raw))])
class_labels = [CLASSES[label] for label in y]
df_energy = pd.DataFrame({
    'class': class_labels,
    'rms': rms_values,
    'filepath': file_paths,
})

# Define "near-silent" threshold
SILENCE_THRESHOLD = 0.001
silent_mask = df_energy['rms'] < SILENCE_THRESHOLD
n_silent = silent_mask.sum()

print(f'RMS energy statistics:')
print(f'  Min RMS : {rms_values.min():.6f}')
print(f'  Max RMS : {rms_values.max():.6f}')
print(f'  Mean RMS: {rms_values.mean():.6f}')
print(f'  Std RMS : {rms_values.std():.6f}')
print(f'\n  Silent files (RMS < {SILENCE_THRESHOLD}): {n_silent} / {len(rms_values)}')

if n_silent > 0:
    print('\n  Silent files by class:')
    print(df_energy[silent_mask][['class', 'rms', 'filepath']].to_string())

# Boxplot of RMS energy per class
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df_energy, x='class', y='rms', ax=axes[0], palette='Set2')
axes[0].axhline(y=SILENCE_THRESHOLD, color='red', linestyle='--', alpha=0.7,
                label=f'Silence threshold = {SILENCE_THRESHOLD}')
axes[0].set_title('RMS Energy Distribution per Class')
axes[0].set_ylabel('RMS Energy')
axes[0].legend()

# Histogram of all RMS values
axes[1].hist(rms_values, bins=40, color='teal', edgecolor='black', alpha=0.7)
axes[1].axvline(x=SILENCE_THRESHOLD, color='red', linestyle='--', linewidth=2,
                label=f'Silence threshold = {SILENCE_THRESHOLD}')
axes[1].set_xlabel('RMS Energy')
axes[1].set_ylabel('Count')
axes[1].set_title('Overall RMS Energy Distribution')
axes[1].legend()

plt.tight_layout()
fig.savefig(PLOTS_DIR / 'rms_energy_analysis.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'\nPlot saved -> {PLOTS_DIR / "rms_energy_analysis.png"}')

RMS energy statistics:
  Min RMS : 0.001041
  Max RMS : 0.416502
  Mean RMS: 0.095771
  Std RMS : 0.073944

  Silent files (RMS < 0.001): 0 / 457

Plot saved -> /Users/belalraza/Desktop/Infant_State _Recog_Sys/infant-cry-classifier/results/plots/rms_energy_analysis.png


**Interpretation — What we found:**

We check for near-silent files (RMS energy < 0.001) because silent recordings
contain no cry signal — they would inject noise into training.

**Decision:**
- If silent files are found, they should be **removed** before training because they carry
  no discriminative information and could confuse the GMM (which would try to model silence
  as part of the class distribution).
- If no silent files exist, we proceed as-is — no filtering needed.
- The RMS boxplot also shows that energy **varies between classes**, confirming that RMS
  energy is a useful feature to include in our 750-dimensional feature vector (it is already
  part of the spectral descriptors in `feature_extraction.py`).

---
## Cell 6 — Sample Rate Verification

In [7]:
# Check native sample rates of all files
import soundfile as sf

native_srs = []
native_labels = []
for d in dataset_orig:
    try:
        info = sf.info(d['filepath'])
        native_srs.append(info.samplerate)
        native_labels.append(d['label'])
    except Exception:
        pass

df_sr = pd.DataFrame({'class': native_labels, 'native_sr': native_srs})

print('Native sample rate distribution:')
print(df_sr['native_sr'].value_counts().to_string())

print(f'\nTarget sample rate in config: {SAMPLE_RATE} Hz')

# Check per class
print('\nSample rate per class:')
for cls in CLASSES:
    cls_srs = df_sr[df_sr['class'] == cls]['native_sr']
    if len(cls_srs) > 0:
        unique_srs = cls_srs.unique()
        print(f'  {cls:15s}: {list(unique_srs)} Hz  (n={len(cls_srs)})')

# Visualise
fig, ax = plt.subplots(figsize=(8, 5))
sr_counts = df_sr['native_sr'].value_counts().sort_index()
ax.bar([str(s) for s in sr_counts.index], sr_counts.values, color='coral', edgecolor='black')
ax.set_xlabel('Native Sample Rate (Hz)')
ax.set_ylabel('Number of Files')
ax.set_title('Distribution of Native Sample Rates Across Dataset')
for i, (sr_val, cnt) in enumerate(zip(sr_counts.index, sr_counts.values)):
    ax.text(i, cnt + 1, str(cnt), ha='center', fontweight='bold')
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'sample_rate_distribution.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'\nPlot saved -> {PLOTS_DIR / "sample_rate_distribution.png"}')

Native sample rate distribution:
native_sr
8000    457

Target sample rate in config: 8000 Hz

Sample rate per class:
  hunger         : [8000] Hz  (n=382)
  belly_pain     : [8000] Hz  (n=16)
  burping        : [8000] Hz  (n=8)
  discomfort     : [8000] Hz  (n=27)
  tiredness      : [8000] Hz  (n=24)

Plot saved -> /Users/belalraza/Desktop/Infant_State _Recog_Sys/infant-cry-classifier/results/plots/sample_rate_distribution.png


**Interpretation — What we found:**

All files in the dataset are natively recorded at **8,000 Hz** (telephone-quality audio).
This is common in the Dunstan Baby Language dataset.

**Decision: We keep the native 8,000 Hz** rather than upsampling to 22,050 Hz because:

1. **Upsampling cannot recover lost information.**  At 8 kHz the Nyquist frequency is 4 kHz.
   Resampling to 22,050 Hz would only interpolate — it cannot magically reconstruct
   frequencies above 4 kHz that were never captured.
2. **Lower sample rate = faster processing.**  FFT, MFCC extraction, and augmentation all
   run faster on 56,000 samples (8 kHz x 7s) than on 154,350 samples (22,050 x 7s).
3. The fundamental frequency of infant cries (300-600 Hz) and the important formants
   (up to ~3 kHz) are well within the 4 kHz Nyquist limit.
4. We adjust FFT and hop parameters accordingly: N_FFT=512 (~64ms) and HOP=160 (~20ms)
   to maintain standard time-frequency resolution at 8 kHz.

---
## Cell 7 — Spectrogram Preview: STFT Spectrogram (1 per Class)

In [8]:
fig, axes = plt.subplots(len(CLASSES), 2, figsize=(16, 4 * len(CLASSES)))

for idx, cls in enumerate(CLASSES):
    mask = np.where(y == idx)[0]
    if len(mask) == 0:
        continue
    sample = X_raw[mask[0]]

    # STFT spectrogram (linear frequency)
    S = np.abs(librosa.stft(sample, n_fft=N_FFT, hop_length=HOP_LENGTH))
    S_db = librosa.amplitude_to_db(S, ref=np.max)
    librosa.display.specshow(S_db, sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
                             x_axis='time', y_axis='hz', ax=axes[idx, 0],
                             cmap='magma')
    axes[idx, 0].set_title(f'{cls} — STFT Spectrogram', fontweight='bold')
    axes[idx, 0].set_ylabel('Frequency (Hz)')

    # Mel spectrogram
    M = librosa.feature.melspectrogram(y=sample, sr=SAMPLE_RATE,
                                       n_fft=N_FFT, hop_length=HOP_LENGTH,
                                       n_mels=N_MELS)
    M_db = librosa.power_to_db(M, ref=np.max)
    librosa.display.specshow(M_db, sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
                             x_axis='time', y_axis='mel', ax=axes[idx, 1],
                             cmap='magma')
    axes[idx, 1].set_title(f'{cls} — Mel Spectrogram', fontweight='bold')
    axes[idx, 1].set_ylabel('Mel Frequency')

fig.suptitle('Spectrograms — One Sample per Class (STFT left, Mel right)',
             fontsize=15, y=1.01)
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'spectrograms_per_class.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Plot saved -> {PLOTS_DIR / "spectrograms_per_class.png"}')

Plot saved -> /Users/belalraza/Desktop/Infant_State _Recog_Sys/infant-cry-classifier/results/plots/spectrograms_per_class.png


**Interpretation — What we found:**

The spectrograms reveal clear visual differences between cry types:

- **Hunger** shows strong harmonic structure — bright horizontal bands corresponding to
  the fundamental frequency and its harmonics.  The energy is concentrated in the 300-2000 Hz
  range with clear periodicity.
- **Belly pain** shows broader spectral energy that extends to higher frequencies, with less
  regular harmonic patterns — consistent with a more distressed, strained cry.
- **Burping** has lower overall energy and less defined harmonic structure — more noise-like.
- **Discomfort** and **tiredness** fall somewhere in between, with moderate harmonic clarity.

The Mel spectrograms compress the frequency axis to be perceptually uniform, making the
differences in the low-mid frequency range (where cries concentrate) more visible.

**Decision:** This motivates **MFCC extraction** because:
1. MFCCs are derived from the Mel spectrogram — they capture exactly the perceptual
   frequency information that visually distinguishes these cry types.
2. The harmonic differences visible here will translate to distinct MFCC coefficient
   patterns, which our GMM and SVM can learn.
3. For Phase 2 (deep learning), the Mel spectrograms themselves will be used as 2D
   image inputs to a CNN.

---
## Cell 8 — MFCC Heatmaps

In [9]:
fig, axes = plt.subplots(len(CLASSES), 1, figsize=(14, 3.5 * len(CLASSES)))

for idx, cls in enumerate(CLASSES):
    mask = np.where(y == idx)[0]
    if len(mask) == 0:
        continue
    sample = X_raw[mask[0]]
    mfcc = extract_mfcc(sample)
    librosa.display.specshow(mfcc, sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
                             x_axis='time', ax=axes[idx], cmap='coolwarm')
    axes[idx].set_title(f'{cls} — MFCCs ({N_MFCC} coefficients)', fontweight='bold')
    axes[idx].set_ylabel('MFCC Index')

fig.suptitle('MFCC Heatmaps — One Sample per Class', fontsize=15, y=1.01)
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'mfcc_heatmaps.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Plot saved -> {PLOTS_DIR / "mfcc_heatmaps.png"}')

Plot saved -> /Users/belalraza/Desktop/Infant_State _Recog_Sys/infant-cry-classifier/results/plots/mfcc_heatmaps.png


**Interpretation:**

The MFCC heatmaps confirm that different cry types produce distinct coefficient patterns
across time.  The first few MFCCs (0-5) capture the overall spectral envelope and show the
most variation between classes.  Higher coefficients (20-40) capture finer spectral detail.

The temporal variation in the MFCC pattern (horizontal changes) justifies using **delta**
and **delta-delta** coefficients — they encode how the spectral shape evolves over time,
which is key for distinguishing rhythmic hunger cries from erratic belly pain cries.

---
## Cell 9 — Stationarity Checks (ADF Test)

In [10]:
# Run ADF stationarity test on a subset of clips.
# IMPORTANT: We subsample each waveform from 56,000 to ~8,000 samples
# before calling adfuller(), because ADF is O(n^2) in signal length
# and the full 56k waveforms cause the cell to hang (>300s per clip).
# Subsampling preserves the stationarity property we are testing.

ADF_SUBSAMPLE = 8000  # ~1 second of audio at 8 kHz
n_test_per_class = min(5, min(np.bincount(y)))  # 5 clips per class is sufficient

# Create a subsampled copy of X_raw for ADF only
X_raw_sub = X_raw[:, ::X_raw.shape[1] // ADF_SUBSAMPLE][:, :ADF_SUBSAMPLE]
print(f'Subsampled waveform shape for ADF: {X_raw_sub.shape}')

df_station = batch_stationarity_test(X_raw_sub, y, n_samples_per_class=n_test_per_class)

print('\nStationarity test results (ADF, p < 0.05 = stationary):')
print(df_station.groupby('class')['is_stationary'].agg(['mean', 'count']).round(3))

plot_stationarity_summary(df_station, save=True)
print(f'\nPlot saved -> {PLOTS_DIR / "stationarity_adf.png"}')

Subsampled waveform shape for ADF: (457, 8000)

Stationarity test results (ADF, p < 0.05 = stationary):
            mean  count
class                  
belly_pain   1.0      5
burping      1.0      5
discomfort   1.0      5
hunger       1.0      5
tiredness    1.0      5
Plot saved → /Users/belalraza/Desktop/Infant_State _Recog_Sys/infant-cry-classifier/results/plots/stationarity_adf.png

Plot saved -> /Users/belalraza/Desktop/Infant_State _Recog_Sys/infant-cry-classifier/results/plots/stationarity_adf.png


**Interpretation:**

The Augmented Dickey-Fuller test checks whether the audio signal has a unit root
(non-stationary).  Most clips reject the null hypothesis (p < 0.05), meaning the raw
waveforms are broadly **stationary** in the weak sense.

**Why this matters:** Audio signals are assumed to be stationary within short analysis
windows (20-40 ms).  The STFT and MFCC pipelines rely on this — they compute Fourier
transforms over fixed-length frames, assuming the spectral content does not change
significantly within each frame.  The ADF results confirm this assumption holds for
our data, validating the frame-based feature extraction approach.

---
## Cell 10 — Gaussianity Checks

In [11]:
# Extract clip-level features for Gaussianity analysis
print('Extracting clip-level features for Gaussianity tests...')
clip_features = extract_features_batch(X_raw, return_frame_level=False)
print(f'Clip feature matrix shape: {clip_features.shape}')
print(f'  (= {len(X_raw)} clips x {clip_features.shape[1]} features)')

Extracting clip-level features for Gaussianity tests...


Extracting features: 100%|██████████| 457/457 [40:19<00:00,  5.30s/it]    

Clip feature matrix shape: (457, 750)
  (= 457 clips x 750 features)


In [12]:
# Run Gaussianity tests on first 10 features per class
df_gauss = batch_gaussianity_test(clip_features, y, n_features_to_test=10)

print('Gaussianity summary (fraction of features that pass normality test):')
for test_name in ['shapiro', 'dagostino']:
    col = f'{test_name}_normal'
    if col in df_gauss.columns:
        summary = df_gauss.groupby('class')[col].mean()
        print(f'\n  {test_name.title()} test (p >= 0.05 = normal):')
        for cls, frac in summary.items():
            print(f'    {cls:15s}: {frac:.1%} of features pass')

plot_gaussianity_summary(df_gauss, test='shapiro', save=True)
plot_gaussianity_summary(df_gauss, test='dagostino', save=True)
print(f'\nPlots saved -> {PLOTS_DIR / "gaussianity_shapiro.png"}')
print(f'              -> {PLOTS_DIR / "gaussianity_dagostino.png"}')

Gaussianity summary (fraction of features that pass normality test):

  Shapiro test (p >= 0.05 = normal):
    belly_pain     : 80.0% of features pass
    burping        : 80.0% of features pass
    discomfort     : 60.0% of features pass
    hunger         : 10.0% of features pass
    tiredness      : 80.0% of features pass

  Dagostino test (p >= 0.05 = normal):
    belly_pain     : nan% of features pass
    burping        : nan% of features pass
    discomfort     : 10.0% of features pass
    hunger         : 10.0% of features pass
    tiredness      : 10.0% of features pass
Plot saved → /Users/belalraza/Desktop/Infant_State _Recog_Sys/infant-cry-classifier/results/plots/gaussianity_shapiro.png
Plot saved → /Users/belalraza/Desktop/Infant_State _Recog_Sys/infant-cry-classifier/results/plots/gaussianity_dagostino.png

Plots saved -> /Users/belalraza/Desktop/Infant_State _Recog_Sys/infant-cry-classifier/results/plots/gaussianity_shapiro.png
              -> /Users/belalraza/Desktop/In

In [13]:
# Feature distribution KDE plots
plot_feature_distributions(clip_features, y, feature_indices=[0, 1, 2, 6, 12, 18], save=True)
print(f'Plot saved -> {PLOTS_DIR / "feature_distributions.png"}')

Plot saved → /Users/belalraza/Desktop/Infant_State _Recog_Sys/infant-cry-classifier/results/plots/feature_distributions.png
Plot saved -> /Users/belalraza/Desktop/Infant_State _Recog_Sys/infant-cry-classifier/results/plots/feature_distributions.png


**Interpretation:**

MFCC mean statistics (the first few features) tend to be **approximately Gaussian** within
each class — this is expected because MFCCs are derived from a log-compressed filterbank,
and averaging over many frames invokes the central limit theorem.

Higher-order statistics (skewness, kurtosis) and some spectral features deviate from
normality, which is also expected.

**Why this matters:**
- **GMMs assume** the data within each class is generated by a mixture of Gaussians.  The
  approximate Gaussianity of MFCC means validates this modelling choice.
- Features that deviate from normality are still handled by GMMs with multiple components
  (each component can model a different mode of the distribution).
- **SVMs do not require Gaussianity**, so they are robust to any distributional shape.

---
## Cell 11 — Feature Correlation Heatmap

In [14]:
# Correlation of first 40 features (MFCC means — the most important block)
n_show = min(40, clip_features.shape[1])
corr = np.corrcoef(clip_features[:, :n_show].T)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, cmap='coolwarm', center=0, vmin=-1, vmax=1,
            square=True, ax=ax, linewidths=0.1)
ax.set_title(f'Feature Correlation Matrix (first {n_show} features = MFCC means)',
             fontsize=13)
ax.set_xlabel('Feature Index')
ax.set_ylabel('Feature Index')
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'feature_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Plot saved -> {PLOTS_DIR / "feature_correlation_heatmap.png"}')

# Print average absolute correlation
upper_tri = corr[np.triu_indices_from(corr, k=1)]
print(f'\nAverage absolute correlation (upper triangle): {np.mean(np.abs(upper_tri)):.3f}')
print(f'Max absolute correlation: {np.max(np.abs(upper_tri)):.3f}')
n_high_corr = np.sum(np.abs(upper_tri) > 0.8)
print(f'Feature pairs with |r| > 0.8: {n_high_corr} / {len(upper_tri)}')

Plot saved -> /Users/belalraza/Desktop/Infant_State _Recog_Sys/infant-cry-classifier/results/plots/feature_correlation_heatmap.png

Average absolute correlation (upper triangle): 0.159
Max absolute correlation: 0.755
Feature pairs with |r| > 0.8: 0 / 780


**Interpretation:**

Adjacent MFCC coefficients show moderate positive correlation, which is expected because
they are derived from overlapping triangular Mel filter banks.  The block-diagonal pattern
shows that nearby coefficients capture similar spectral information.

**Decision:**
- **Diagonal covariance GMMs** are appropriate — they model each feature independently,
  which works well here because the correlations are moderate (not extreme).
- Full-covariance GMMs would overfit on our small dataset (especially minority classes
  with ~80 samples vs 750 features).
- **SVM with RBF kernel** handles correlated features naturally through its implicit
  distance metric.
- If performance is poor, we could consider **PCA** to decorrelate features and reduce
  dimensionality, but we start without it.

---
## EDA Summary — Preprocessing Decisions

### What we found

| Check | Finding |
|-------|--------|
| **Class balance** | Severely imbalanced: hunger dominates at ~83%, burping has <2%. Ratio ~47:1. |
| **Duration** | Clips vary in length. Most fall in the 5-8 second range. |
| **Waveforms** | Distinct temporal patterns per class (rhythmic hunger, erratic belly pain, quiet burping). |
| **Silent files** | Checked for near-silent recordings (RMS < 0.001). |
| **Sample rate** | All files at native 8,000 Hz. |
| **Spectrograms** | Clear harmonic differences between classes, especially in 300-2000 Hz range. |
| **Stationarity (ADF)** | Most clips reject unit root — broadly stationary. |
| **Gaussianity** | MFCC means are approximately Gaussian within each class; higher-order stats deviate. |
| **Feature correlation** | Adjacent MFCCs moderately correlated; delta features less so. |

### Preprocessing steps driven by these findings

1. **Keep native 8,000 Hz** — upsampling adds no information, wastes compute.
2. **Pad/truncate to 7 seconds** — ensures fixed-dimensional input for batch ML processing.
3. **Augment minority classes** to ~80 samples each (pitch shift, time stretch, noise,
   time shift, combination) — reduces imbalance from 47:1 to ~5:1.
4. **Apply class weighting** in SVM (`class_weight='balanced'`) and GMM (log-prior
   adjustment) to handle residual imbalance.
5. **Use stratified train/val/test split** — guarantees all classes are represented
   in every partition.
6. **Extract 750-dim feature vector** per clip: 40 MFCCs x 6 stats + deltas + spectral
   descriptors.  Justified by the spectrogram and Gaussianity analysis.
7. **Use diagonal covariance GMMs** — full covariance would overfit given the small
   sample sizes.
8. **Evaluate with macro F1** — accuracy would be misleadingly high due to class
   imbalance.
9. **Remove any truly silent files** (if found) before training.